# Partie II — Autoencodeur

## 0. Préparation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch :", torch.__version__)
print("Device :", device)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°1 - Mise en oeuvre du Framework PyTorch\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_14624\1533387746.py", line 1, in <module>
    import torch
  File "c:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°1 - Mise en oeuvre du Framework PyTorch\.venv\Lib\site-packages\torch\__init__.py", line 3312, in <module>
    from torch import (
  File "c:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°1 - Mise en oeuvre du Framework PyTorch\.venv\Lib\site-packages\torch\export\__init__.py", line 11, in <module>
    from torch.fx.passes.infra.pass_base import PassResult
  File "c:\Users\user\Desktop\5IAD G1 

In [2]:
# Pixels dans [0, 1] : adaptés à la sortie Sigmoid du décodeur
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print("Train :", len(train_dataset))
print("Test  :", len(test_dataset))

NameError: name 'transforms' is not defined

## 3. Idée générale

Un autoencodeur apprend à reconstruire son entrée :

$x \rightarrow Encoder \rightarrow z \rightarrow Decoder \rightarrow \hat x$, avec $\hat x \approx x$

Le vecteur $z$ est une **représentation latente** de l'entrée. L'apprentissage devient $x \rightarrow x$ :
les labels ne sont plus nécessaires.

L'encodeur compresse 784 pixels en `latent_dim = 2` valeurs ; le décodeur reconstruit l'image et se termine
par `Sigmoid` pour produire des pixels entre 0 et 1, comme les images d'origine.

**Question du cours : si vous deviez décrire une image avec seulement quelques nombres, que faudrait-il conserver ?**
Les informations essentielles pour reconnaître l'image et en reconstruire une approximation : la forme générale,
les contours, la position, l'orientation. Les détails fins et le bruit peuvent être laissés de côté. Pour un chiffre
manuscrit, quelques nombres pourraient décrire la présence d'une boucle, l'inclinaison, la courbure et l'épaisseur
des traits, plutôt que chaque pixel. C'est le rôle du code latent $z$, que le réseau apprend lui-même : une
coordonnée latente ne correspond pas forcément à une caractéristique identifiable.

**Influence de la dimension latente**

| Dimension latente | Effet possible |
|---|---|
| Trop faible | Perte d'informations, reconstruction dégradée |
| Intermédiaire | Compromis entre compression et fidélité |
| Très élevée | Reconstruction facilitée, mais représentation peu structurée (le réseau peut approcher l'identité) |

Ici, `latent_dim = 2` est volontairement très faible : la reconstruction sera floue, mais on pourra afficher
l'espace latent dans un plan.

**Étapes d'entraînement** : sélectionner un lot, calculer les représentations latentes, reconstruire les entrées,
calculer la perte de reconstruction, rétropropager, mettre à jour les paramètres.

In [3]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        x_hat = self.decoder(z)
        return x_hat.view(-1, 1, 28, 28)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat

ae = Autoencoder(latent_dim=2).to(device)
ae

NameError: name 'nn' is not defined

## 4. Loss de reconstruction

$\mathcal{L}_{rec}=\frac{1}{N}\sum_{i=1}^{N}(x_i-\hat{x}_i)^2$ avec `MSELoss`.
On ignore les labels (`for x, _ in loader`).

In [4]:
ae_criterion = nn.MSELoss()
ae_optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)

def train_autoencoder(model, loader, epochs=5):
    model.train()
    history = []

    for epoch in range(epochs):
        total_loss = 0

        for x, _ in loader:
            x = x.to(device)

            x_hat = model(x)
            loss = ae_criterion(x_hat, x)

            ae_optimizer.zero_grad()
            loss.backward()
            ae_optimizer.step()

            total_loss += loss.item()

        history.append(total_loss / len(loader))
        print(f"Epoch {epoch+1}/{epochs} - Loss = {history[-1]:.4f}")

    return history

ae_history = train_autoencoder(ae, train_loader, epochs=5)

NameError: name 'nn' is not defined

## 5. Visualisation des reconstructions

Première ligne : images originales. Deuxième ligne : images reconstruites.

In [5]:
ae.eval()

x, _ = next(iter(test_loader))
x = x[:10].to(device)

with torch.no_grad():
    x_hat = ae(x)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.2))

for i in range(10):
    axes[0, i].imshow(x[i].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, i].axis("off")

    axes[1, i].imshow(x_hat[i].cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[1, i].axis("off")

axes[0, 0].set_title("Original", loc="left")
axes[1, 0].set_title("Reconstruit", loc="left")

plt.tight_layout()
plt.show()

NameError: name 'ae' is not defined

Avec seulement 2 valeurs latentes, la forme générale du chiffre est conservée mais les détails sont perdus :
les reconstructions sont floues et certains chiffres proches (4/9, 3/5) se confondent.

## 6. Visualisation de l'espace latent

Avec `latent_dim = 2`, chaque image MNIST devient un point du plan. On colore les points par leur label.

In [6]:
ae.eval()

zs = []
ys = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        z = ae.encode(x)

        zs.append(z.cpu())
        ys.append(y)

Z = torch.cat(zs).numpy()
Y = torch.cat(ys).numpy()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(Z[:, 0], Z[:, 1], c=Y, cmap="tab10", s=4)
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Espace latent appris par l'autoencodeur")
plt.colorbar(scatter, ticks=range(10))
plt.show()

NameError: name 'ae' is not defined

Les chiffres forment des groupes, alors que l'autoencodeur n'a jamais vu les labels. Mais l'espace est
irrégulier : groupes de tailles très différentes, zones vides entre eux, valeurs non centrées.

## Génération à partir d'un chiffre connu

On encode toutes les images d'un chiffre, on fait la moyenne de leurs vecteurs latents, puis on décode cette moyenne.

In [7]:
@torch.no_grad()
def generer_chiffre(model, dataloader, chiffre):
    if chiffre not in range(10):
        raise ValueError("Le chiffre doit être compris entre 0 et 9.")

    device = next(model.parameters()).device
    model.eval()

    somme_z = None
    nombre = 0

    for images, labels in dataloader:
        # Sélection des images du chiffre demandé
        images = images[labels == chiffre]

        if images.shape[0] == 0:
            continue

        z = model.encode(images.to(device))

        if somme_z is None:
            somme_z = z.sum(dim=0, keepdim=True)
        else:
            somme_z += z.sum(dim=0, keepdim=True)

        nombre += z.shape[0]

    if nombre == 0:
        raise ValueError(f"Aucune image du chiffre {chiffre} trouvée.")

    z_moyen = somme_z / nombre
    return model.decode(z_moyen).cpu()

NameError: name 'torch' is not defined

In [8]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2))

for chiffre, ax in enumerate(axes):
    image = generer_chiffre(ae, train_loader, chiffre=chiffre)
    ax.imshow(image[0, 0].numpy(), cmap="gray")
    ax.set_title(str(chiffre))
    ax.axis("off")

plt.suptitle("Décodage du vecteur latent moyen de chaque chiffre")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 7. Limite d'un autoencodeur classique

L'AE apprend des représentations utiles, mais rien ne garantit que l'espace latent soit continu, régulier,
facile à échantillonner ou distribué selon une loi connue. Le test ci-dessous décode des points tirés au
hasard selon une loi normale $\mathcal N(0, I)$.

In [9]:
with torch.no_grad():
    z = torch.randn(10, 2, device=device)
    images = ae.decode(z).cpu()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, image in zip(axes, images):
    ax.imshow(image[0], cmap="gray")
    ax.axis("off")
plt.suptitle("Points latents tirés au hasard : images peu réalistes")
plt.show()

NameError: name 'torch' is not defined

Beaucoup de ces images ne ressemblent pas à des chiffres : les points tirés tombent dans des zones que l'encodeur
n'utilise pas. Cette difficulté motive le **Variational Autoencoder** (Partie III).